# HilbertMaze

[Codility task](https://app.codility.com/programmers/trainings/2/hilbert_maze/) - Exercise 2, 2016 Contest

The maze of size `N` lives on a grid of `M x M` cells, `M = 2^(N+1) + 1` (up to `2^26 + 1 = 67,108,865` per side). It is built recursively from four mazes of size `N - 1` (bottom-left rotated clockwise, bottom-right rotated counter-clockwise, top ones unrotated) plus three extra rocks. `solution(N, A, B, C, D)` returns the minimum number of steps from `(A, B)` to `(C, D)` moving north, south, west or east through empty cells.

| N | start | treasure | result |
|---|-------|----------|--------|
| 1 | (2, 1) | (3, 4) | 8 |
| 2 | (2, 5) | (6, 6) | 7 |
| 3 | (6, 6) | (10, 13) | 39 |

`x` grows to the right and `y` upwards. The README describes the exact layout (reconstructed from the pictures of the statement).

## Idea

A breadth-first search is impossible: the grid has up to `4.5 * 10^15` cells. Use the structure.

**What the maze is.** Cells with both coordinates odd are the centres of the `4^N` small blocks and are always rocks; the rocks joining two consecutive blocks complete a **Hilbert curve** (`2 * 4^N - 1` rocks in total). Every other cell is empty. In other words the empty cells are the *walls of a perfect maze* whose only corridor is the Hilbert curve: a grid of lines (corners at even/even cells, edges in between) from which the edges crossed by the curve have been removed.

**Consequence (checked below by counting).** The graph of empty cells has `2 (2^N + 1)^2` vertices and the same number of edges: exactly **one cycle**, the border of the grid. Everything else is made of trees, each hanging from a single border cell.

**Distances.**

- two cells hanging from *different* border cells: walk up to the border, take the short way round the border (perimeter `4 * 2^(N+1)`), walk down into the other tree;
- two cells in the *same* tree: the unique path inside the tree.

**Recursion.** In the maze of size `N` (with `mid = 2^N`) the **skeleton** is the border plus the middle cross (the row `y = mid` and the column `x = mid`). The three extra rocks `(1, mid)`, `(M - 2, mid)` and `(mid, mid + 1)` cut it so that it is just the border cycle with two trees attached: a **T** (the column from `(mid, 0)` up to the centre and the row through the centre) hanging from `(mid, 0)`, and a short column hanging from `(mid, M - 1)`. Every other empty cell lies strictly inside one quarter, and in that quarter (a maze of size `N - 1`) it hangs from some cell of the quarter's own border, which is a skeleton cell of the big maze. So:

1. `attach(N, cell)` = the border cell where the cell hangs and its depth: if the cell is on the skeleton, read it off the T; otherwise recurse into its quarter (undoing the rotation), map the quarter's border cell back, and add the skeleton depth. It takes `N` steps.
2. `dist(N, u, v)`: if the border cells differ, use the border formula; if they agree but the skeleton cells differ, use the distance in the T; if both hang from the same skeleton cell of the same quarter, recurse into that quarter; otherwise the two branches meet at that cell.

Time O(N^2) in the worst case (O(N) per level), memory O(N). Everything is verified against a literal construction plus BFS for `N <= 7`.

In [ ]:
def border_size(level):
    return 2 ** (level + 1)                       # M - 1


def on_border(level, x, y):
    s = border_size(level)
    return x in (0, s) or y in (0, s)


def ring_position(level, x, y):
    """Position of a border cell along the border, counter-clockwise from (0, 0)."""
    s = border_size(level)
    if y == 0:
        return x
    if x == s:
        return s + y
    if y == s:
        return 2 * s + (s - x)
    return 3 * s + (s - y)


def ring_distance(level, p, q):
    around = 4 * border_size(level)
    d = abs(ring_position(level, *p) - ring_position(level, *q))
    return min(d, around - d)


def quarter_of(level, x, y):
    """Quarter of a cell that is strictly inside one, and its coordinates in the quarter's own (unrotated) frame."""
    mid = 2 ** level
    if x < mid and y < mid:                       # bottom-left: the quarter is rotated clockwise
        return "BL", (mid - y, x)
    if x > mid and y < mid:                       # bottom-right: rotated counter-clockwise
        return "BR", (y, mid - (x - mid))
    if x < mid:                                   # top-left
        return "TL", (x, y - mid)
    return "TR", (x - mid, y - mid)               # top-right


def from_quarter(level, quarter, lx, ly):
    """Inverse of quarter_of: a cell of a quarter's own frame -> coordinates in the big maze."""
    mid = 2 ** level
    if quarter == "BL":
        return ly, mid - lx
    if quarter == "BR":
        return mid + (mid - ly), lx
    if quarter == "TL":
        return lx, ly + mid
    return lx + mid, ly + mid


def skeleton_gate(level, x, y):
    """Cell of the border or of the middle cross -> (border cell where it hangs, distance to it)."""
    s = border_size(level)
    mid = 2 ** level
    if on_border(level, x, y):
        return (x, y), 0
    if x == mid and y <= mid:                     # column of the T
        return (mid, 0), y
    if y == mid:                                  # row of the T
        return (mid, 0), mid + abs(x - mid)
    return (mid, s), s - y                        # upper column (x == mid, y >= mid + 2)


def skeleton_distance(level, c1, c2):
    """Distance between two skeleton cells of the same tree (the T, or the upper column)."""
    mid = 2 ** level
    (x1, y1), (x2, y2) = c1, c2
    row1 = y1 == mid and x1 != mid
    row2 = y2 == mid and x2 != mid
    if row1 and row2:
        return abs(x1 - x2)
    if row1:
        return abs(x1 - mid) + abs(mid - y2)
    if row2:
        return abs(x2 - mid) + abs(mid - y1)
    return abs(y1 - y2)                           # both on the column x == mid


def attach(level, x, y):
    """Border cell where the cell hangs, and the distance to it (inside a maze of this level)."""
    if level == 0 or on_border(level, x, y):
        return (x, y), 0
    mid = 2 ** level
    if x == mid or y == mid:
        return skeleton_gate(level, x, y)
    quarter, (lx, ly) = quarter_of(level, x, y)
    (cx, cy), depth = attach(level - 1, lx, ly)
    gate, extra = skeleton_gate(level, *from_quarter(level, quarter, cx, cy))
    return gate, depth + extra


def locate(level, x, y):
    """(skeleton cell where the cell hangs, depth to it, (quarter, local cell) or None for a skeleton cell)."""
    mid = 2 ** level
    if on_border(level, x, y) or x == mid or y == mid:
        return (x, y), 0, None
    quarter, (lx, ly) = quarter_of(level, x, y)
    (cx, cy), depth = attach(level - 1, lx, ly)
    return from_quarter(level, quarter, cx, cy), depth, (quarter, (lx, ly))


def dist(level, u, v):
    if u == v:
        return 0
    if level == 0:                                # 3 x 3 maze: only the border ring of 8 cells
        return ring_distance(0, u, v)
    cu, du, ku = locate(level, *u)
    cv, dv, kv = locate(level, *v)
    gu, up_u = skeleton_gate(level, *cu)
    gv, up_v = skeleton_gate(level, *cv)
    if gu != gv:                                  # different trees: go round the border
        return du + up_u + dv + up_v + ring_distance(level, gu, gv)
    if cu != cv:                                  # same tree, different skeleton cells
        return du + dv + skeleton_distance(level, cu, cv)
    if ku is not None and kv is not None and ku[0] == kv[0]:
        return dist(level - 1, ku[1], kv[1])      # same skeleton cell, same quarter: look inside it
    return du + dv                                # the two branches meet at that skeleton cell


def solution(N, A, B, C, D):
    return dist(N, (A, B), (C, D))

## Official examples

In [ ]:
assert solution(1, 2, 1, 3, 4) == 8
assert solution(2, 2, 5, 6, 6) == 7
assert solution(3, 6, 6, 10, 13) == 39
print("examples OK")

## Reference: the maze built literally

The reference constructs the grid exactly as the statement says (a `3 x 3` maze with a central rock for size 0, four rotated copies overlapping on the middle row and column, three extra rocks) and runs a BFS. Rotating clockwise maps `(x, y)` to `(y, n - 1 - x)`; counter-clockwise maps it to `(n - 1 - y, x)`.

In [ ]:
from collections import deque


def rotate_clockwise(grid):
    n = len(grid)
    out = [[False] * n for _ in range(n)]
    for y in range(n):
        for x in range(n):
            if grid[y][x]:
                out[n - 1 - x][y] = True
    return out


def rotate_counterclockwise(grid):
    n = len(grid)
    out = [[False] * n for _ in range(n)]
    for y in range(n):
        for x in range(n):
            if grid[y][x]:
                out[x][n - 1 - y] = True
    return out


def build_maze(N):
    """grid[y][x] is True for a rock. Quarters overlap on the shared row and column."""
    if N == 0:
        grid = [[False] * 3 for _ in range(3)]
        grid[1][1] = True
        return grid
    sub = build_maze(N - 1)
    q = len(sub)
    m = 2 * q - 1
    mid = q - 1
    grid = [[False] * m for _ in range(m)]
    parts = [(rotate_clockwise(sub), 0, 0), (rotate_counterclockwise(sub), mid, 0), (sub, 0, mid), (sub, mid, mid)]
    for part, ox, oy in parts:
        for y in range(q):
            for x in range(q):
                if part[y][x]:
                    grid[oy + y][ox + x] = True
    for x, y in ((1, mid), (m - 2, mid), (mid, mid + 1)):         # the three additional rocks
        grid[y][x] = True
    return grid


def bfs_distances(grid, source):
    m = len(grid)
    dist_map = {source: 0}
    queue = deque([source])
    while queue:
        x, y = queue.popleft()
        for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
            if 0 <= nx < m and 0 <= ny < m and not grid[ny][nx] and (nx, ny) not in dist_map:
                dist_map[(nx, ny)] = dist_map[(x, y)] + 1
                queue.append((nx, ny))
    return dist_map


assert bfs_distances(build_maze(1), (2, 1))[(3, 4)] == 8
assert bfs_distances(build_maze(2), (2, 5))[(6, 6)] == 7
assert bfs_distances(build_maze(3), (6, 6))[(10, 13)] == 39
print("the literal construction reproduces the three examples")

## The structure behind the idea

Counted on the literal mazes for `N = 1..6`: the number of rocks is `2 * 4^N - 1` (a Hilbert curve), the number of empty cells is `2 (2^N + 1)^2`, the graph of empty cells has as many edges as vertices (**one cycle**) and no `2 x 2` empty block exists. The maze is also left-right symmetric.

In [ ]:
for N in range(1, 7):
    grid = build_maze(N)
    m = len(grid)
    empty = {(x, y) for y in range(m) for x in range(m) if not grid[y][x]}
    rocks = m * m - len(empty)
    edges = sum(((x + 1, y) in empty) + ((x, y + 1) in empty) for x, y in empty)
    squares = sum(((x + 1, y) in empty) and ((x, y + 1) in empty) and ((x + 1, y + 1) in empty) for x, y in empty)
    assert rocks == 2 * 4**N - 1
    assert len(empty) == 2 * (2**N + 1) ** 2
    assert edges == len(empty) and squares == 0                       # connected + one cycle
    assert len(bfs_distances(grid, (0, 0))) == len(empty)             # connected
    assert all(grid[y][x] == grid[y][m - 1 - x] for y in range(m) for x in range(m))
print("rocks, cells, a single cycle, connectivity and left-right symmetry verified for N = 1..6")

## `is_rock` without building the grid

To pick random empty cells for `N = 25` the same recursion gives a rock test in O(N): the border and the cross are empty except for the three extra rocks, and any other cell is a rock iff it is a rock in its quarter. It is verified against the literal grids.

In [ ]:
def is_rock(level, x, y):
    if level == 0:
        return (x, y) == (1, 1)
    s = border_size(level)
    mid = 2 ** level
    if on_border(level, x, y):
        return False
    if (x, y) in ((1, mid), (s - 1, mid), (mid, mid + 1)):
        return True
    if x == mid or y == mid:
        return False
    quarter, (lx, ly) = quarter_of(level, x, y)
    return is_rock(level - 1, lx, ly)


for N in range(1, 7):
    grid = build_maze(N)
    m = len(grid)
    assert all(is_rock(N, x, y) == grid[y][x] for y in range(m) for x in range(m)), N
print("is_rock agrees with the literal construction for N = 1..6")

## Exhaustive and sampled comparison with BFS

All pairs of empty cells for `N = 1, 2, 3`; for `N = 4` all targets from 150 random sources; for `N = 5, 6, 7` 25 random sources against 1,500 random targets each.

In [ ]:
import random

rng = random.Random(10)


def check(N, sources, targets_per_source):
    grid = build_maze(N)
    m = len(grid)
    empty = [(x, y) for y in range(m) for x in range(m) if not grid[y][x]]
    chosen = empty if sources is None else rng.sample(empty, sources)
    pairs = 0
    for s in chosen:
        reference = bfs_distances(grid, s)
        targets = empty if targets_per_source is None else rng.sample(empty, targets_per_source)
        for t in targets:
            assert solution(N, s[0], s[1], t[0], t[1]) == reference[t], (N, s, t)
            pairs += 1
    return pairs


for N, sources, targets in [(1, None, None), (2, None, None), (3, None, None), (4, 150, None),
                            (5, 25, 1500), (6, 25, 1500), (7, 25, 1500)]:
    print(f"N = {N}: {check(N, sources, targets):>7} pairs agree with BFS")

## Large mazes (`N = 25`, a grid of 67 million cells per side)

There is no reference at this size, so the checks are consistency properties that a wrong recursion would break: symmetry `d(u, v) = d(v, u)`, the triangle inequality, left-right mirror symmetry of the maze, `d >= Manhattan distance`, and the bound promised by the statement (the result is smaller than `2^31 - 1`).

In [ ]:
import time

N = 25
M = 2 ** (N + 1) + 1


def random_empty(level):
    s = border_size(level)
    while True:
        x, y = rng.randint(0, s), rng.randint(0, s)
        if not is_rock(level, x, y):
            return x, y


def d(u, v, level=N):
    return solution(level, u[0], u[1], v[0], v[1])


worst = 0
for _ in range(3000):
    u, v, w = random_empty(N), random_empty(N), random_empty(N)
    duv, dvu = d(u, v), d(v, u)
    assert duv == dvu
    assert duv >= abs(u[0] - v[0]) + abs(u[1] - v[1])                  # can never beat the Manhattan distance
    assert d(u, w) <= duv + d(v, w)                                    # triangle inequality
    mirror = lambda p: (M - 1 - p[0], p[1])
    assert d(mirror(u), mirror(v)) == duv                              # the maze is left-right symmetric
    assert duv < 2**31 - 1
    worst = max(worst, duv)
print("3000 random triples pass the consistency checks; largest distance seen:", f"{worst:,}")

# the border is a cycle of 4 * 2^26 cells: opposite corners are half way round
assert d((0, 0), (2 ** (N + 1), 2 ** (N + 1))) == 2 ** (N + 2)
assert d((0, 0), (2 ** (N + 1), 0)) == 2 ** (N + 1)
print("border distances OK")

## Performance

In [ ]:
queries = [(random_empty(N), random_empty(N)) for _ in range(20_000)]
start = time.perf_counter()
total = sum(d(u, v) for u, v in queries)
elapsed = time.perf_counter() - start
print(f"20,000 queries at N = 25 in {elapsed:.2f}s ({elapsed / 20_000 * 1e6:.0f} microseconds per query)")

## Notes

- Do not stop at "BFS works for the examples": with `N = 25` the grid has about `4.5 * 10^15` cells. Look for the structure that makes the problem small.
- The rocks being a Hilbert curve means the empty space is the wall system of a maze with a single corridor: one cycle (the border) plus trees. Distances in "a cycle with trees" are easy.
- The recursion is exact because trees only touch the rest of the graph through one cell, so shortest paths between skeleton cells never enter a tree.
- Validation strategy: build the object literally for small sizes, compare **all pairs**, then rely on mathematical consistency checks (symmetry, triangle inequality, mirror symmetry, lower bound) for the sizes that cannot be enumerated.